# SR3 Diffusion on BioSR Microscopy Images

Turning blurry widefield microscope images into sharp SIM-like images with a diffusion model (SR3).

**Before running anything:** go to Runtime → Change runtime type → pick **T4 GPU**.

Then run the cells top to bottom. The trained model gets saved to Google Drive, so nothing's lost if Colab disconnects.

**Already trained the model?** Run steps 1 and 2, then skip straight to step 8.

## 1. Connect Google Drive + check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# should say Tesla T4. if it errors, you're not on a GPU runtime
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
Uses the zip from your Google Drive if it's there (top level of My Drive). If not, you'll get a button to upload it from your computer.
Works with either `sr3_microscopy.zip` or `sr3-microscopy-diffusion.zip`.

In [ ]:
import os, glob, shutil, subprocess
from google.colab import files

os.chdir('/content')
CODE = '/content/sr3_microscopy'

# look for the zip in Drive first
zips = sorted(glob.glob('/content/drive/MyDrive/sr3*microscopy*.zip'), key=os.path.getmtime)
if zips:
    z = zips[-1]
else:
    print('No zip found in My Drive, pick it from your computer:')
    z = '/content/' + list(files.upload())[0]
print('using', z)

# unzip and put the code in one place
shutil.rmtree('/content/unzipped', ignore_errors=True)
subprocess.run(['unzip', '-oq', z, '-d', '/content/unzipped'], check=True)
src = os.path.dirname(glob.glob('/content/unzipped/**/train.py', recursive=True)[0])
# move the dataset out of the way while we replace the code
if os.path.exists(f'{CODE}/biosr_dataset'):
    shutil.move(f'{CODE}/biosr_dataset', '/content/biosr_dataset')
shutil.rmtree(CODE, ignore_errors=True)
shutil.move(src, CODE)

# if the dataset is already downloaded, keep it next to the code
if os.path.exists('/content/biosr_dataset'):
    shutil.move('/content/biosr_dataset', f'{CODE}/biosr_dataset')

os.chdir(CODE)
!pip -q install tifffile scikit-image
print('code ready:', os.path.exists('configs/biosr.json'), '| dataset here:', os.path.exists('biosr_dataset'))

## 3. Download the dataset
**For training:** download everything (about 6 GB, takes ~6 min).
**Only testing an already trained model?** Set `FULL = False` to grab just the 1,500 test images (much faster).

In [ ]:
FULL = True   # change to False if you only need the test images

T = 'biosr_dataset/BioSR/Microtubules'
if not os.path.exists('biosr_dataset'):
    if FULL:
        !git clone --depth 1 https://github.com/DeepTrackAI/biosr_dataset
    else:
        !git clone --depth 1 --filter=blob:none --sparse https://github.com/DeepTrackAI/biosr_dataset
        !cd biosr_dataset && git sparse-checkout set BioSR/Microtubules/test_gt BioSR/Microtubules/test_wf

# count the files in each folder
for d in sorted(glob.glob(f'{T}/*')):
    print(d.split('/')[-1], len(os.listdir(d)))

## 4. Look at a few examples (top: blurry input, bottom: sharp target)

In [ ]:
import tifffile
import matplotlib.pyplot as plt

names = sorted(os.listdir(f'{T}/training_gt'))[:4]
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for i, n in enumerate(names):
    ax[0, i].imshow(tifffile.imread(f'{T}/training_wf/{n}').squeeze(), cmap='gray')
    ax[0, i].set_title('widefield (input)')
    ax[1, i].imshow(tifffile.imread(f'{T}/training_gt/{n}').squeeze(), cmap='gray')
    ax[1, i].set_title('SIM (target)')
for a in ax.flat:
    a.axis('off')
plt.tight_layout()

## 5. Training settings
Medium-size model (~15M parameters) so it fits on the free T4. Saves to Drive every 2,500 steps.
60,000 steps took me about 4 hours at ~4 steps/sec.

In [ ]:
import json

RUN = '/content/drive/MyDrive/sr3_runs/biosr'   # where everything gets saved

cfg = json.load(open('configs/biosr.json'))
cfg['model']['base_channels'] = 32
cfg['train'].update(out_dir=RUN, total_steps=60000, batch_size=16,
                    sample_every=2500, ckpt_every=2500, num_workers=2)
json.dump(cfg, open('configs/biosr_colab.json', 'w'), indent=2)
print('settings saved, results will go to', RUN)

## 6. Train
Takes a few minutes to load the images before the first line prints. Then you'll see `step ... | loss ... | it/s` every 200 steps.

**If Colab disconnects:** run steps 1, 2, 3 and 5 again, then use the resume cell instead.

In [ ]:
!python train.py --config configs/biosr_colab.json

In [ ]:
# resume from the last save (only use this after a disconnect)
!python train.py --config configs/biosr_colab.json --resume {RUN}/last.pt

## 7. Check progress
Shows the newest preview. Each row: blurry input | model output | real SIM.
Training has to be stopped (or finished) for this to run.

In [ ]:
from IPython.display import Image, display

latest = sorted(glob.glob(f'{RUN}/samples/*.png'))[-1]
print(latest.split('/')[-1])
display(Image(latest, width=500))

## 8. Test set numbers
Runs the trained model on the 150 test images it never saw, at 3 noise levels (01 = noisiest, 05 = medium, 09 = cleanest).
For each level it prints PSNR and SSIM for the blurry input vs the model. Higher = closer to the real SIM image.

Takes roughly 5 minutes per level on a T4. It prints the table at the end of each level, so it'll look stuck for a bit, that's normal.

In [ ]:
RUN = '/content/drive/MyDrive/sr3_runs/biosr'
T = 'biosr_dataset/BioSR/Microtubules'

for lvl in ['01', '05', '09']:
    print(f'\n===== noise level {lvl} =====')
    !python evaluate.py --ckpt {RUN}/last.pt --steps 50 --beta_end 0.1 --n 150 \
        --hr_dir {T}/test_gt --lr_dir {T}/test_wf/level_{lvl} --out {RUN}/test_level_{lvl}

## 9. Results table + pictures for GitHub
Puts everything in one folder in Drive: `sr3_runs/biosr/for_github`.
Copy the table into the README / report.

In [ ]:
import pandas as pd
from IPython.display import Image, display

OUT = f'{RUN}/for_github'
os.makedirs(OUT, exist_ok=True)

# one table with all 3 noise levels
rows = []
for lvl in ['01', '05', '09']:
    d = f'{RUN}/test_level_{lvl}'
    if not os.path.exists(f'{d}/metrics.csv'):
        print('missing level', lvl, '(run step 8 first)')
        continue
    m = pd.read_csv(f'{d}/metrics.csv').mean()
    rows.append({'Noise level': lvl,
                 'Input PSNR': round(m.input_psnr, 2), 'Model PSNR': round(m.sr3_psnr, 2),
                 'Input SSIM': round(m.input_ssim, 3), 'Model SSIM': round(m.sr3_ssim, 3)})
    shutil.copy(f'{d}/preview.png', f'{OUT}/test_noise_level_{lvl}.png')

print(pd.DataFrame(rows).to_string(index=False))

# show the test pictures
for f in sorted(glob.glob(f'{OUT}/test_noise_level_*.png')):
    print(f.split('/')[-1])
    display(Image(f, width=400))